# 02 · Preparación de los datos — Dry Bean Dataset
**CRISP-DM fase 3: Preparación de los datos** · Sprint 1 · Backlog: PB-02b



| **Entrada**| `drybean_raw.csv` — datos crudos(Salida del EDA) |
| **Salida** | `drybean_clean.csv` — datos limpios y formateados |
| **Salida** | `labelencoder.joblib` — codificador de la variable objetivo |

El notebook sigue las siguientes tareas: **seleccionar, limpiar, construir, integrar y formatear** los datos. Cada decisión queda justificada, aunque la decisión sea "no hacer nada".

## 1. Entorno

In [31]:
import pandas as pd

# Cargar el archivo CSV
ruta_archivo = "Datos/Datos Crudos/drybean_raw.csv"
df = pd.read_csv(ruta_archivo)

print("Dataset cargado")
print("Dimensiones:", df.shape)

Python: 3.13.18
pandas: 2.2.3
numpy: 2.4.6
scikit-learn: 1.6.1
joblib: 1.6.0


In [32]:
from google.colab import drive
drive.mount("/content/drive")

RUTA = "/content/drive/MyDrive/"

df = pd.read_csv(RUTA + "drybean_raw.csv")
print("Datos crudos:", df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datos crudos: (13611, 17)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


## 2. Seleccionar datos
**Tarea:** decidir qué columnas y filas se usarán en el modelado.

**Decisión: se conservan las 16 variables predictoras y todas las filas válidas.**
- El EDA encontró 21 pares de variables con |r| > 0,9 (por ejemplo, `Area` y `ConvexArea`). Aun así, no se eliminan en esta etapa: la redundancia afecta de forma distinta a cada técnica (pesa más en modelos lineales que en modelos de árboles), así que la decisión de reducir variables se deja para el modelado.


In [33]:
predictoras = [c for c in df.columns if c != "Class"]
print(f"{len(predictoras)} variables predictoras:")
print(predictoras)

16 variables predictoras:
['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4']


## 3. Limpiar datos
**Tarea:** corregir o eliminar problemas de calidad detectados en el EDA.

### 3.1 Duplicados
El EDA encontró 68 filas idénticas en las 17 columnas, todas de la clase HOROZ. Se interpretan como el mismo grano registrado dos veces y **se eliminan**: si se dejan, una misma fila podría quedar en entrenamiento y en prueba a la vez, y eso inflaría las métricas.

In [34]:
registros_antes = len(df)
duplicados = df.duplicated().sum()
print("Duplicados encontrados:", duplicados)
print("Clases de los duplicados:", df[df.duplicated()]["Class"].value_counts().to_dict())

df = df.drop_duplicates().reset_index(drop=True)

print(f"Registros: {registros_antes:,} → {len(df):,}")

Duplicados encontrados: 68
Clases de los duplicados: {'HOROZ': 68}
Registros: 13,611 → 13,543


### 3.2 Valores nulos
El EDA no encontró nulos. Se verifica de nuevo; **no se requiere imputación**.

In [35]:
print("Nulos totales:", df.isna().sum().sum())

Nulos totales: 0


### 3.3 Valores atípicos
**Decisión: no se eliminan.** Según la regla IQR, cerca del 22 % de las filas tiene algún valor "atípico", pero el EDA mostró que los atípicos de tamaño son los 522 granos BOMBAY (más algunos CALI y BARBUNYA): granos reales de variedades grandes, no errores de medición. Eliminarlos borraría casi toda la clase BOMBAY.

## 4. Construir datos
**Tarea:** crear variables nuevas (derivadas) o registros nuevos.

**Decisión: no se construyen variables nuevas.** Las 16 variables ya describen el tamaño y la forma del grano, y varias de ellas ya son variables derivadas (por ejemplo, `Compactness` y los `ShapeFactor`). No hay información adicional (como el color) para crear variables que aporten algo distinto.

Los registros sintéticos que creará **SMOTE** se generan en el modelado, solo sobre entrenamiento (ver sección 7).

## 5. Integrar datos
**Tarea:** combinar datos de varias fuentes.

**No aplica:** el proyecto usa una sola fuente (UCI Machine Learning Repository, id 602).

## 6. Formatear datos
**Tarea:** cambios de formato que no alteran el significado de los datos.

### 6.1 Nombres de columnas
El archivo de UCI trae `AspectRation` (error tipográfico) y `roundness` (en minúscula). Se renombran a `AspectRatio` y `Roundness`para corregir el error y unificar el formato de los nombres

In [36]:
df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})
print(df.columns.tolist())

['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']


### 6.2 Codificación de la variable objetivo (LabelEncoder)
La variable objetivo `Class` está en texto (BARBUNYA, BOMBAY, ...). Varios algoritmos de clasificación requieren que la clase sea un número entero consecutivo desde 0, así que se codifica para que el dataset quede listo para cualquier técnica que se seleccione en el modelado. `LabelEncoder` asigna un número a cada variedad, en orden alfabético. Los números son solo etiquetas: no implican ningún orden entre variedades.

Se agrega la columna `Class_cod` y se conserva `Class` con el nombre original, para que el archivo siga siendo legible.

**¿Hay fuga de información?** No: el codificador solo traduce nombres a números, no aprende nada de la distribución de los datos. Por eso puede ajustarse aquí, antes de la partición.

In [37]:
label_encoder = LabelEncoder()
df["Class_cod"] = label_encoder.fit_transform(df["Class"])

pd.DataFrame({
    "clase": label_encoder.classes_,
    "codigo": range(len(label_encoder.classes_)),
})

,clase,codigo
0,BARBUNYA,0
1,BOMBAY,1
2,CALI,2
3,DERMASON,3
4,HOROZ,4
5,SEKER,5
6,SIRA,6


## 7. Transformaciones que se aplicarán en el modelado
Estas dos transformaciones son parte de la preparación, pero **no se ejecutan en este notebook**. Ambas aprenden de los datos, así que deben ajustarse **solo con el conjunto de entrenamiento**, después de la partición train/test. La partición es la tarea *Generar diseño de prueba* de la fase de Modelado, por eso se ejecutan en otro notebook.

| Transformación | Problema que resuelve (hallazgo del EDA) | Por qué después de la partición |
| --- | --- | --- |
| **SMOTE** | Clases desbalanceadas: DERMASON tiene 6,8 veces más registros que BOMBAY | Si se aplica antes, habría granos sintéticos en prueba creados a partir de granos de entrenamiento |
| **StandardScaler** | Escalas muy distintas: `Area` ≈ 53.000 y `ShapeFactor2` ≈ 0,002 | Si se ajusta con todos los datos, las medias y desviaciones "verían" el conjunto de prueba |

En el notebook 03 ambas irán dentro de un *pipeline*, así se aplican correctamente también en cada partición de la validación cruzada.

## 8. Verificación del resultado

In [38]:
verificacion = {
    "registros": len(df),
    "columnas": df.shape[1],
    "duplicados": int(df.drop(columns="Class_cod").duplicated().sum()),
    "nulos": int(df.isna().sum().sum()),
    "clases": df["Class"].nunique(),
}
assert verificacion["duplicados"] == 0 and verificacion["nulos"] == 0
verificacion

{'registros': 13543, 'columnas': 18, 'duplicados': 0, 'nulos': 0, 'clases': 7}

In [39]:
distribucion = pd.DataFrame({
    "registros": df["Class"].value_counts(),
    "porcentaje": (df["Class"].value_counts(normalize=True) * 100).round(1),
})
distribucion

,registros,porcentaje
Class,,
DERMASON,3546,26.2
SIRA,2636,19.5
SEKER,2027,15.0
HOROZ,1860,13.7
CALI,1630,12.0
BARBUNYA,1322,9.8
BOMBAY,522,3.9


Tras quitar duplicados, HOROZ pasa de 1.928 a 1.860 registros. Las demás clases no cambian y el desbalance se mantiene (se tratará con SMOTE en el modelado).

## 9. Guardar en Drive

In [40]:
df.to_csv(RUTA + "drybean_clean.csv", index=False)
joblib.dump(label_encoder, RUTA + "labelencoder.joblib")

print("Guardados en", RUTA)
print(" - drybean_clean.csv:", df.shape)
print(" - labelencoder.joblib:", list(label_encoder.classes_))

Guardados en /content/drive/MyDrive/
 - drybean_clean.csv: (13543, 18)
 - labelencoder.joblib: ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']


## 10. Resumen de la preparación

| Tarea CRISP-DM | Decisión | Registros afectados |
| --- | --- | --- |
| Seleccionar | Se conservan las 16 variables y todas las filas válidas | — |
| Limpiar: duplicados | Eliminados (todos HOROZ) | 13.611 → 13.543 (−68) |
| Limpiar: nulos | No hay; sin imputación | 0 |
| Limpiar: atípicos | Se conservan (granos BOMBAY reales) | 0 |
| Construir | Sin variables nuevas | — |
| Integrar | No aplica (una sola fuente) | — |
| Formatear: nombres | `AspectRation` → `AspectRatio`, `roundness` → `Roundness` | — |
| Formatear: clase | LabelEncoder → columna `Class_cod` (0 a 6) | — |
| SMOTE | Definido aquí; se aplica en el modelado, solo a entrenamiento | — |
| StandardScaler | Definido aquí; se ajusta en el modelado, solo con entrenamiento | — |
